In [ ]:
# v1: CORAL ordinal probe — fine-tune from the teacher-label 0.89 checkpoint.
# (Direct paths only — no recursive ** globs. Justin's 2026-10-09 fix:
#  recursive globs over /kaggle/input/** walk 36GB of Phase 0 cache.)
import glob, os

VERSION = "phase2-coral-v1 (2026-10-09: CORAL ordinal probe — fine-tune from teacher-label effnet_best.pt ep14 [LB 0.89]; binary heads primary on severity-merged labels v1; CORAL auxiliary λ=0.4, threshold weights [1.0, 0.5, 0.5]; 5 epochs)"
print(f"VERSION: {VERSION}")
PROBE = True              # True: tiny subset, validate pipeline. False: full 5-epoch probe run.
PROBE_N_STUDIES = 8

# --- attachments (direct paths; fail-fast asserts run BEFORE any heavy work) ---
BASE_CKPT      = "/kaggle/input/datasets/sridharshivashankar/effnet-best-ckpt/effnet_best.pt"
SEV_BATCH_GLOB = "/kaggle/input/datasets/sridharshivashankar/severity-batches-v1/*.json"
MERGED_CSV_PATH  = "/kaggle/input/datasets/sridharshivashankar/severity-merged-labels-v1/merged_labels_v1.csv"
TEACHER_CSV_PATH = "/kaggle/input/datasets/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv"

print("=== fail-fast attachment checks (top of Cell 0) ===")
assert os.path.exists(BASE_CKPT), \
    f"FATAL: base ckpt missing: {BASE_CKPT} -- attach the effnet-best-ckpt dataset"
_sev_files = sorted(glob.glob(SEV_BATCH_GLOB))
assert len(_sev_files) == 9, \
    f"FATAL: expected 9 severity batch JSONs, found {len(_sev_files)} -- attach severity-batches-v1"
print(f"  base ckpt: OK ({BASE_CKPT})")
print(f"  severity batches: OK ({len(_sev_files)} JSONs)")
if os.path.exists(MERGED_CSV_PATH):
    LABEL_CSV, LABEL_SOURCE = MERGED_CSV_PATH, "severity-merged v1 (teacher base + Gemini severity)"
elif os.path.exists(TEACHER_CSV_PATH):
    LABEL_CSV, LABEL_SOURCE = TEACHER_CSV_PATH, "teacher (0.8927 vs gold)"
else:
    raise RuntimeError("FATAL: no label CSV found -- attach the severity-merged-labels-v1 dataset")
print(f"  labels: {LABEL_CSV} [{LABEL_SOURCE}]")
print("fail-fast: ALL GREEN")

# --- data ---
IMAGE_SIZE = 288
N_SLICES = 8
PLANES = ['axial', 'coronal', 'sagittal']

# --- model ---
BACKBONE_NAME = 'efficientnet_b3'
PRETRAINED = False        # backbone weights come from BASE_CKPT, not ImageNet
DROP_PATH_RATE = 0.1
ATTN_DIM = 128
N_FINDINGS = 12

# --- CORAL ordinal heads (audit 2026-10-09: ordinal grades were never expert-validated,
# so binary presence heads stay PRIMARY on the 100%-validated labels; CORAL is a
# low-weight AUXILIARY with full weight only on threshold-0 (presence), reduced on
# t1/t2 (unvalidated grade granularity)) ---
SEV_FINDINGS = ['Synovitis', 'PF OA', 'Medial OA', 'Lateral OA']
SEV_IDX = [8, 6, 4, 5]          # LABELS indices of the 4 severity findings
N_THRESH = 3                    # none|mild|moderate|severe -> 3 CORAL thresholds
CORAL_LAMBDA = 0.4              # PLAN-14DAY §Day-3: CORAL auxiliary weight
CORAL_THRESH_W = [1.0, 0.5, 0.5]
GRADE_TO_THRESH = {"none": (0, 0, 0), "mild": (1, 0, 0),
                   "moderate": (1, 1, 0), "severe": (1, 1, 1)}

# --- training: 5-epoch fine-tune from the converged 0.89 ckpt ---
BATCH_SIZE = 2
ACCUM_STEPS = 8
EPOCHS = 5
LR_BACKBONE = 3e-5            # gentle: backbone already converged
LR_HEAD = 3e-4                # binary heads: already trained, fine-tune
LR_CORAL = 1e-3               # coral heads: fresh params, need full LR
WEIGHT_DECAY = 0.02
WARMUP_FRAC = 0.10
GRAD_CLIP = 1.0
EMA_DECAY = 0.998
SWA_LAST = 3
USE_AMP = True
FREEZE_BN = True
GOLD_UPWEIGHT = 8.0

# --- ASL loss (same as CoAtNet / v2.3) ---
ASL_GAMMA_POS = 0.0
ASL_GAMMA_NEG = 4.0
ASL_MARGIN = 0.05

# --- folds (same SEED/fold as v2.3 for apples-to-apples comparison) ---
N_FOLDS = 5
VAL_FOLD = 0
SEED = 42

# --- io ---
CKPT_DIR = '/kaggle/working/coral_ckpt'

# --- baseline to beat (teacher-label effnet_best.pt ep14, LB 0.89) ---
BASELINE_GOLD_MACRO = 0.8628

LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
          'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']

print('CONFIG DUMP')
for k in ['PROBE','IMAGE_SIZE','N_SLICES','BACKBONE_NAME','PRETRAINED','DROP_PATH_RATE',
          'BATCH_SIZE','ACCUM_STEPS','EPOCHS','LR_BACKBONE','LR_HEAD','LR_CORAL',
          'CORAL_LAMBDA','CORAL_THRESH_W','WEIGHT_DECAY','WARMUP_FRAC','GRAD_CLIP',
          'EMA_DECAY','SWA_LAST','USE_AMP','FREEZE_BN','GOLD_UPWEIGHT',
          'ASL_GAMMA_POS','ASL_GAMMA_NEG','ASL_MARGIN',
          'N_FOLDS','VAL_FOLD','SEED','CKPT_DIR','BASELINE_GOLD_MACRO']:
    print(f'  {k} = {globals()[k]}')


In [ ]:
# ================= SETUP: imports, device, seeds, attachments =================
import os, sys, json, time, math, random, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print(f'torch {torch.__version__} | cuda: {torch.cuda.is_available()}',
      f' | gpus: {torch.cuda.device_count()}')
assert torch.cuda.is_available(), 'FATAL: no GPU'
NGPU = torch.cuda.device_count()
for i in range(NGPU):
    print(f'  gpu{i}: {torch.cuda.get_device_name(i)}')
DEVICE = torch.device('cuda:0')

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

# ---- attachment check (bounded walk, no recursive ** glob) ----
def find_under(root, name, max_depth=6):
    hits = []
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.count(os.sep)
        if depth - root.count(os.sep) > max_depth:
            dirnames[:] = []
            continue
        if name in filenames or name in dirnames:
            hits.append(os.path.join(dirpath, name))
    return hits

print('\nattachment scan (/kaggle/input):')
for req in ['manifest.json', 'merged_labels_v1.csv', 'effnet_best.pt']:
    hits = find_under('/kaggle/input', req)
    print(f'  {req}: {len(hits)} hit(s)')
    for h in hits[:3]:
        print(f'    {h}')

MANIFEST_PATHS = find_under('/kaggle/input', 'manifest.json')
assert MANIFEST_PATHS, 'FATAL: manifest.json not found — attach phase0runA/B/C'
assert os.path.exists(BASE_CKPT), f'FATAL: base ckpt missing: {BASE_CKPT}'

TRAIN_CSV_CANDS = find_under('/kaggle/input', 'train.csv')
TRAIN_CSV = TRAIN_CSV_CANDS[0] if TRAIN_CSV_CANDS else None
print(f'manifests: {len(MANIFEST_PATHS)} | label csv: {LABEL_CSV} [{LABEL_SOURCE}]')
print(f'train.csv: {TRAIN_CSV} (for gold labels + priors)')
os.makedirs(CKPT_DIR, exist_ok=True)

assert os.path.exists(LABEL_CSV), f'FATAL: chosen label CSV missing: {LABEL_CSV}'


In [ ]:
# ================= CORE MATH (tested locally; do not modify without re-testing) =================
def asl_loss(logits, targets, gamma_pos=ASL_GAMMA_POS, gamma_neg=ASL_GAMMA_NEG,
             margin=ASL_MARGIN, eps=1e-8):
    """logits, targets: (B, 12). Soft targets supported. Returns (B, 12) element losses."""
    p = torch.sigmoid(logits).clamp(eps, 1 - eps)
    loss_pos = -((1 - p) ** gamma_pos) * torch.log(p)
    p_m = (p - margin).clamp(min=0).clamp(eps, 1 - eps)
    loss_neg = -((p_m) ** gamma_neg) * torch.log(1 - p_m)
    return targets * loss_pos + (1 - targets) * loss_neg


def auc_score(y_true, y_score):
    y_true = np.asarray(y_true); y_score = np.asarray(y_score)
    pos = y_score[y_true == 1]; neg = y_score[y_true == 0]
    if len(pos) == 0 or len(neg) == 0:
        return None
    n1, n0 = len(pos), len(neg)
    labs = np.concatenate([np.ones(n1), np.zeros(n0)])
    sc = np.concatenate([pos, neg])
    order = np.argsort(sc, kind='mergesort')
    s_sorted = sc[order]
    ranks = np.empty(len(sc))
    i = 0
    while i < len(sc):
        j = i
        while j < len(sc) and s_sorted[j] == s_sorted[i]:
            j += 1
        ranks[order[i:j]] = (i + 1 + j) / 2.0
        i = j
    r1 = ranks[labs == 1].sum()
    return (r1 - n1 * (n1 + 1) / 2.0) / (n1 * n0)


def macro_auc(y_true_2d, y_score_2d, labels=LABELS):
    aucs = {}
    for j, lab in enumerate(labels):
        a = auc_score(y_true_2d[:, j], y_score_2d[:, j])
        aucs[lab] = a
    valid = [a for a in aucs.values() if a is not None]
    return (sum(valid) / len(valid) if valid else None), aucs


def scanner_group_key(study_uid, n_parts=8):
    return '.'.join(study_uid.split('.')[:n_parts])


def assign_folds(study_uids, n_folds=N_FOLDS, seed=SEED):
    rng = np.random.RandomState(seed)
    groups = {}
    for uid in study_uids:
        groups.setdefault(scanner_group_key(uid), []).append(uid)
    keys = sorted(groups.keys())
    rng.shuffle(keys)
    fold_of = {}
    for i, k in enumerate(keys):
        for uid in groups[k]:
            fold_of[uid] = i % n_folds
    return fold_of, groups


# self-test
_lt = torch.tensor([[0.0, 2.0, -2.0]]); _tg = torch.tensor([[1.0, 1.0, 0.0]])
_L = asl_loss(_lt, _tg)
assert _L[0, 1] < _L[0, 0] and _L[0, 2] < _L[0, 0], 'ASL focusing broken'
assert abs(auc_score([0, 0, 1, 1], [0.1, 0.2, 0.8, 0.9]) - 1.0) < 1e-9
assert abs(auc_score([0, 1], [0.5, 0.5]) - 0.5) < 1e-9
assert auc_score([1, 1], [0.5, 0.6]) is None
print('core math self-test: ALL GREEN')


In [ ]:
# ================= DATA LOADING: cache index =================
class ShardStore:
    def __init__(self, manifest_paths):
        self.studies = {}
        self._z = {}
        for mp in manifest_paths:
            cache_dir = str(Path(mp).parent)
            man = json.load(open(mp))
            for uid, entry in man['studies'].items():
                if uid in self.studies:
                    raise RuntimeError(f'duplicate study {uid} across caches')
                self.studies[uid] = (cache_dir, entry)
        print(f'indexed {len(self.studies)} studies from {len(manifest_paths)} manifests')

    def get_plane(self, uid, plane):
        pid = os.getpid()
        cache_dir, entry = self.studies[uid]
        shard = entry['shard']
        key = (pid, cache_dir, shard)
        z = self._z.get(key)
        if z is None:
            for k in [k for k in self._z if k[0] != pid]:
                try:
                    self._z.pop(k).close()
                except Exception:
                    pass
            z = np.load(os.path.join(cache_dir, shard))
            self._z[key] = z
            while len(self._z) > 4:
                k = next(iter(self._z))
                try:
                    self._z.pop(k).close()
                except Exception:
                    pass
        return z[f'{uid}/{plane}']

    def planes_of(self, uid):
        return list(self.studies[uid][1]['planes'].keys())


store = ShardStore(MANIFEST_PATHS)
from collections import Counter
pc = Counter()
for uid in store.studies:
    pc[tuple(sorted(store.planes_of(uid)))] += 1
print('plane-set distribution:')
for k, v in pc.most_common():
    print(f'  {k}: {v}')
n_missing = sum(1 for uid in store.studies if len(store.planes_of(uid)) < 3)
print(f'studies with <3 planes: {n_missing} (must be tolerated, never dropped)')


In [ ]:
# ================= LABELS: teacher soft targets + gold override =================
lab_df = pd.read_csv(LABEL_CSV, encoding='latin-1')
print(f'label csv: {lab_df.shape} | source: {LABEL_SOURCE}')

# Header normalization: match case/whitespace-insensitively, reorder to LABELS.
# (Public tables occasionally ship headers like 'acl' or stray whitespace; a
# bare assert would die with zero context. This fails LOUD instead.)
def _norm(c):
    return str(c).strip().casefold()
_norm_map = {}
for c in lab_df.columns:
    n = _norm(c)
    if n in _norm_map:
        raise RuntimeError(f'FATAL: duplicate label column after normalization: {c!r}')
    _norm_map[n] = c
_expected = ['StudyInstanceUID'] + LABELS
_missing = [e for e in _expected if _norm(e) not in _norm_map]
if _missing:
    raise RuntimeError(
        f'FATAL: label CSV is missing columns {sorted(set(_missing))}; '
        f'actual header: {list(lab_df.columns)}')
lab_df = lab_df[[_norm_map[_norm(e)] for e in _expected]]
lab_df.columns = _expected
print(f'  header normalized: StudyInstanceUID + {len(LABELS)} findings')

assert lab_df[LABELS].notna().all().all(), 'FATAL: NaN in label csv'
assert ((lab_df[LABELS] >= 0) & (lab_df[LABELS] <= 1)).all().all(), 'FATAL: labels outside [0,1]'

# Check: are the 58 gold rows verbatim 0/1 in the teacher table?
# (verbatim copies poison cross-validation; fine for final training)
gold = {}
priors = None
if TRAIN_CSV and os.path.exists(TRAIN_CSV):
    tr = pd.read_csv(TRAIN_CSV, encoding='latin-1', usecols=['StudyInstanceUID'] + LABELS)
    gm = tr[LABELS].notna().all(axis=1)
    gdf = tr[gm]
    print(f'gold studies in train.csv: {len(gdf)}')
    for _, r in gdf.iterrows():
        gold[r['StudyInstanceUID']] = r[LABELS].values.astype(np.float32)
    priors = gdf[LABELS].mean().values.astype(np.float32)
    # verbatim check
    _lab_idx = lab_df.set_index('StudyInstanceUID')
    _verbatim = 0
    for uid in gold:
        if uid in _lab_idx.index:
            lv = _lab_idx.loc[uid, LABELS].values
            if np.all(np.isin(lv, [0.0, 1.0])) and np.array_equal(lv, gold[uid]):
                _verbatim += 1
    print(f'  gold rows verbatim 0/1 in teacher table: {_verbatim}/{len(gold)}')
    if _verbatim > len(gold) * 0.5:
        print('  *** WARNING: >50% of gold rows are verbatim copies — DO NOT trust val AUC, use for final training only ***')
else:
    print('WARNING: train.csv not found — gold override disabled')
    priors = lab_df[LABELS].mean().values.astype(np.float32)
print('priors:', dict(zip(LABELS, np.round(priors, 3))))

# build per-study (targets[12], loss_mask[12], sample_weight, is_gold)
# Teacher table: 0.5 = "report does not address" (25.4% of cells in v1).
# Confidence-weighted loss mask w = 2*|p-0.5| (the dataset author's recipe):
#   - 0.5 cells -> weight 0: truly ignored, not trained toward any value
#   - confident cells (0.1/0.9) -> ~0.8, hard 0/1 -> 1.0
# (Without this, ASL with gamma_neg=4 minimizes the 0.5-target loss at
# p ~ 0.68, actively biasing ~25% of cells toward 0.68 predictions.)
# Fallback (weak_labels): use prior-based abstain masking from v12.
lab_idx = lab_df.set_index('StudyInstanceUID')
IS_TEACHER = 'teacher' in LABEL_SOURCE
EPS_ABSTAIN = 1e-6
label_pack = {}
n_gold_hit = n_zero_w = n_abstain = 0
for uid in store.studies:
    if uid in gold:
        label_pack[uid] = (gold[uid], np.ones(12, np.float32), GOLD_UPWEIGHT, True)
        n_gold_hit += 1
    else:
        w = lab_idx.loc[uid, LABELS].values.astype(np.float32)
        if IS_TEACHER:
            mask = (2.0 * np.abs(w - 0.5)).astype(np.float32)
            n_zero_w += int((mask == 0).sum())
        else:
            mask = (np.abs(w - priors) > EPS_ABSTAIN).astype(np.float32)
            n_abstain += int((mask == 0).sum())
        label_pack[uid] = (w, mask, 1.0, False)
print(f'label packs: {len(label_pack)} | gold override: {n_gold_hit} | '
      f'teacher zero-weight (0.5) cells: {n_zero_w} | weak abstained: {n_abstain}')
assert n_gold_hit == len(gold), 'some gold studies missing from cache!'

# ================= CORAL targets: ordinal severity grades =================
# (Binary presence heads above stay PRIMARY on the 100%-validated merged labels.
#  These ordinal targets train the low-weight CORAL auxiliary heads only.)
def _parse_grade(raw):
    # str().lower(): None -> 'none', exactly like merge-severity cell 3 (verified safe).
    g = str(raw).lower().strip()
    if g in GRADE_TO_THRESH:
        return GRADE_TO_THRESH[g], 1.0
    return (0, 0, 0), 0.0  # unknown string -> masked out, never trained

sev = {}
for fp in _sev_files:                      # direct paths from Cell 0 (no ** glob)
    with open(fp) as f:
        data = json.load(f)
    if isinstance(data, dict):
        data = list(data.values())
    for d in data:
        uid = d["StudyInstanceUID"]
        assert uid not in sev, f'FATAL: duplicate severity UID {uid}'
        sev[uid] = d
print(f'severity grades: {len(sev)} studies')
assert len(sev) == 4276, f'FATAL: expected 4276 severity grades, got {len(sev)}'

# severity JSON key -> LABELS index (must match KneeEffNetCORAL.SEV_IDX order)
SEV_KEY_TO_IDX = {"synovitis": 8, "pfoa": 6, "medial_oa": 4, "lateral_oa": 5}
assert [SEV_KEY_TO_IDX[k] for k in ["synovitis", "pfoa", "medial_oa", "lateral_oa"]] == SEV_IDX

n_sev_hit = n_sev_masked = 0
for uid in list(label_pack.keys()):
    t, m, w, g = label_pack[uid]
    ct = np.zeros((4, N_THRESH), np.float32)
    cm = np.zeros(4, np.float32)
    if uid in sev:
        n_sev_hit += 1
        s = sev[uid]
        for i, sk in enumerate(["synovitis", "pfoa", "medial_oa", "lateral_oa"]):
            th, mk = _parse_grade(s.get(sk))
            ct[i] = th
            cm[i] = mk
            n_sev_masked += (mk == 0)
    label_pack[uid] = (t, m, w, g, ct, cm)
print(f'studies with severity grades: {n_sev_hit}/{len(label_pack)} | '
      f'masked grade cells: {n_sev_masked}')
# distribution sanity (expect mostly none, ~20% mild, rare severe)
_thr = np.stack([lp[4] for lp in label_pack.values()])   # (N,4,3)
for i, name in enumerate(SEV_FINDINGS):
    d = _thr[:, i, :].sum(1)
    hist = [(d == k).sum() for k in range(4)]
    print(f'  {name}: none={hist[0]} mild={hist[1]} moderate={hist[2]} severe={hist[3]}')
assert n_gold_hit == len(gold), 'some gold studies missing from cache!'



In [ ]:
# ================= FOLDS: seeded random (UID-prefix proxy collapses) =================
all_uids = sorted(store.studies.keys())
fold_of, groups = assign_folds(all_uids)
import random as _r
if len(groups) < 2:
    print(f'WARNING: only {len(groups)} scanner-proxy group(s); FALLBACK to seeded random folds')
    _rng = _r.Random(SEED)
    _shuf = sorted(all_uids)
    _rng.shuffle(_shuf)
    fold_of = {uid: (idx % N_FOLDS) for idx, uid in enumerate(_shuf)}
    groups = {f'random-fold-{f}': [u for u in all_uids if fold_of[u] == f] for f in range(N_FOLDS)}
print(f'{len(groups)} groups -> {N_FOLDS} folds')
for f in range(N_FOLDS):
    members = [u for u in all_uids if fold_of[u] == f]
    g_gold = sum(1 for u in members if label_pack[u][3])
    print(f'  fold {f}: {len(members)} studies, {g_gold} gold')
train_uids = [u for u in all_uids if fold_of[u] != VAL_FOLD]
val_uids = [u for u in all_uids if fold_of[u] == VAL_FOLD]
print(f'train: {len(train_uids)} | val: {len(val_uids)} (fold {VAL_FOLD})')
if PROBE:
    train_uids = train_uids[:PROBE_N_STUDIES]
    val_uids = val_uids[:max(4, PROBE_N_STUDIES // 4)]
    print(f'PROBE subset -> train: {len(train_uids)} | val: {len(val_uids)}')


In [ ]:
# ================= DATASET: heavy augmentation (PLAN.md §6.6, no flips) =================
def sample_slices(vol, n):
    S = vol.shape[0]
    if S == n:
        return vol
    idx = np.linspace(0, S - 1, n).round().astype(int)
    return vol[idx]


class KneeDataset(Dataset):
    def __init__(self, uids, store, label_pack, image_size=IMAGE_SIZE,
                 n_slices=N_SLICES, augment=False):
        self.uids = uids
        self.store = store
        self.label_pack = label_pack
        self.image_size = image_size
        self.n_slices = n_slices
        self.augment = augment
        self.rng = np.random.RandomState(SEED)

    def __len__(self):
        return len(self.uids)

    def _augment_slice(self, img):
        # img: float32 (H, W) in [0, 1]. NO flips (medial != lateral).
        # ALL rng outputs cast to float32 — float64 leaks crash the GPU forward.
        r = self.rng
        if r.rand() < 0.9:  # p=0.9 heavy augmentation
            # gain (brightness)
            if r.rand() < 0.5:
                g = np.float32(0.85 + 0.30 * r.rand())
                img = np.clip(img * g, 0, 1).astype(np.float32)
            # contrast
            if r.rand() < 0.5:
                c = np.float32(0.8 + 0.45 * r.rand())
                img = np.clip((img - 0.5) * c + 0.5, 0, 1).astype(np.float32)
            # gamma
            if r.rand() < 0.5:
                gm = np.float32(0.7 + 0.7 * r.rand())
                img = np.clip(np.power(img, gm), 0, 1).astype(np.float32)
            # gaussian noise
            if r.rand() < 0.3:
                img = np.clip(img + r.randn(*img.shape).astype(np.float32) * np.float32(0.02),
                              0, 1).astype(np.float32)
            # cutout: 20-50% side rectangle -> 0
            if r.rand() < 0.3:
                h, w = img.shape
                ch = int(h * (0.2 + 0.3 * r.rand()))
                cw = int(w * (0.2 + 0.3 * r.rand()))
                y0 = int(r.rand() * max(1, h - ch))
                x0 = int(r.rand() * max(1, w - cw))
                img = img.copy()
                img[y0:y0+ch, x0:x0+cw] = 0
        return img.astype(np.float32)

    def _affine_vol(self, vol):
        # vol: (S, H, W) float32. Rotation ±15°, zoom 0.90-1.15, shift ±8%.
        # Single affine grid — laterality-safe (no mirroring).
        r = self.rng
        ang = float(r.uniform(-15, 15)) * float(np.pi) / 180.0
        ca, sa = float(np.cos(ang)), float(np.sin(ang))
        zoom = float(0.90 + 0.25 * r.rand())
        tx = float(r.uniform(-0.08, 0.08))
        ty = float(r.uniform(-0.08, 0.08))
        # theta maps output -> input: [zoom*cos, -zoom*sin, tx; zoom*sin, zoom*cos, ty]
        theta = torch.tensor([[[zoom*ca, -zoom*sa, tx],
                               [zoom*sa,  zoom*ca, ty]]], dtype=torch.float32)
        t = torch.from_numpy(vol).unsqueeze(1).float()
        grid = F.affine_grid(theta.expand(t.size(0), -1, -1), t.size(),
                             align_corners=False).to(t.dtype)
        t = F.grid_sample(t, grid, mode='bilinear', padding_mode='zeros',
                          align_corners=False)
        return t.squeeze(1).numpy().astype(np.float32)

    def __getitem__(self, i):
        uid = self.uids[i]
        planes = {}
        mask = []
        have = set(self.store.planes_of(uid))
        for p in PLANES:
            if p in have:
                vol = self.store.get_plane(uid, p)  # (S, 336, 336) uint8
                vol = sample_slices(vol, self.n_slices).astype(np.float32) / 255.0
                if self.augment:
                    vol = np.stack([self._augment_slice(s) for s in vol])
                    if self.rng.rand() < 0.5:
                        vol = self._affine_vol(vol)
                t = torch.from_numpy(vol).unsqueeze(1)  # (S,1,H,W) float32
                t = F.interpolate(t, size=(self.image_size, self.image_size),
                                  mode='bilinear', align_corners=False)
                vol = t.squeeze(1).numpy().astype(np.float32)
                planes[p] = vol
                mask.append(1.0)
            else:
                planes[p] = np.zeros((self.n_slices, self.image_size,
                                      self.image_size), dtype=np.float32)
                mask.append(0.0)
        targets, loss_mask, sample_w, is_gold, coral_t, coral_m = self.label_pack[uid]
        planes3 = {p: np.repeat(v[:, None, :, :], 3, axis=1).astype(np.float32)
                   for p, v in planes.items()}
        return {
            'uid': uid,
            'planes': {p: torch.from_numpy(v) for p, v in planes3.items()},
            'plane_mask': torch.tensor(mask, dtype=torch.float32),
            'targets': torch.from_numpy(targets.astype(np.float32)),
            'loss_mask': torch.from_numpy(loss_mask.astype(np.float32)),
            'weight': torch.tensor(float(sample_w), dtype=torch.float32),
            'is_gold': torch.tensor(bool(is_gold)),
            'coral_targets': torch.from_numpy(coral_t.astype(np.float32)),
            'coral_mask': torch.from_numpy(coral_m.astype(np.float32)),
        }


# dataset self-test
_ds = KneeDataset(train_uids[:4], store, label_pack, augment=True)
_s = _ds[0]
print('sample keys:', sorted(_s.keys()))
for p in PLANES:
    print(f'  {p}: {tuple(_s["planes"][p].shape)}')  # expect (8, 3, 288, 288)
print(f'  plane_mask: {_s["plane_mask"].tolist()}')
print(f'  targets: {_s["targets"].shape} | loss_mask sum: {_s["loss_mask"].sum().item():.0f}/12')
print(f'  coral_targets: {tuple(_s["coral_targets"].shape)} | coral_mask sum: {_s["coral_mask"].sum().item():.0f}/4')
assert _s['coral_targets'].shape == (4, N_THRESH) and _s['coral_mask'].shape == (4,)
assert _s['planes']['axial'].shape == (N_SLICES, 3, IMAGE_SIZE, IMAGE_SIZE)
assert _s['planes']['axial'].dtype == torch.float32
_miss = [u for u in train_uids if len(store.planes_of(u)) < 3]
if _miss:
    _d2 = KneeDataset(_miss[:1], store, label_pack)
    _s2 = _d2[0]
    pm = _s2['plane_mask']
    for p, m in zip(PLANES, pm.tolist()):
        if m == 0:
            assert (_s2['planes'][p] == 0).all(), f'{p} not zeroed'
    print(f'missing-plane study {_miss[0]}: mask {pm.tolist()} -> zeroed OK')
print('dataset self-test: ALL GREEN')


In [ ]:
# ================= MODEL: EfficientNet-B3 2.5D + attention MIL + CORAL ordinal heads ==============
import timm

print('timm', timm.__version__)
assert BACKBONE_NAME in timm.list_models(), f'FATAL: {BACKBONE_NAME} not in this timm version'
print(f'creating {BACKBONE_NAME} (pretrained={PRETRAINED} — weights come from BASE_CKPT)...')


class GatedAttentionMIL(nn.Module):
    def __init__(self, feat_dim, attn_dim=ATTN_DIM):
        super().__init__()
        self.V = nn.Linear(feat_dim, attn_dim)
        self.U = nn.Linear(feat_dim, attn_dim)
        self.w = nn.Linear(attn_dim, 1)

    def forward(self, h):  # h: (B, S, D)
        a = self.w(torch.tanh(self.V(h)) * torch.sigmoid(self.U(h)))
        a = F.softmax(a, dim=1)
        return (a * h).sum(dim=1), a.squeeze(-1)


class CoralHead(nn.Module):
    """Rank-consistent ordinal head (CORAL): 3 thresholds, one shared weight,
    ordered biases b1 >= b2 >= b3 BY CONSTRUCTION, so the predicted probabilities
    satisfy P(>=mild) >= P(>=moderate) >= P(>=severe) for every study.
    (b_ord = [b0, b0 - cumsum(softplus(b[1:]))]; softplus >= 0 guarantees ordering.)"""
    def __init__(self, feat_dim, n_thresh=N_THRESH):
        super().__init__()
        self.w = nn.Linear(feat_dim, 1)
        self.b = nn.Parameter(torch.zeros(n_thresh))

    def forward(self, h):  # h: (B, D) -> (B, 3)
        b0 = self.b[0]
        dec = F.softplus(self.b[1:])                      # >= 0, always
        b_ord = torch.cat([b0.unsqueeze(0), b0 - torch.cumsum(dec, 0)])
        return self.w(h) + b_ord                          # (B, 3)


class KneeEffNetCORAL(nn.Module):
    # Attribute names backbone/mil_heads/heads are IDENTICAL to the v2.x 0.89 ckpt
    # (load_base_ckpt asserts this). Only coral_heads is new.
    SEV_IDX = SEV_IDX  # [8, 6, 4, 5] = Synovitis, PF OA, Medial OA, Lateral OA

    def __init__(self, backbone_name=BACKBONE_NAME, attn_dim=ATTN_DIM,
                 n_findings=N_FINDINGS, pretrained=PRETRAINED,
                 drop_path_rate=DROP_PATH_RATE):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, drop_path_rate=drop_path_rate)
        print(f'backbone init OK: {backbone_name} pretrained={pretrained}')
        D = self.backbone.num_features
        print(f'backbone feature dim: {D}')
        self.mil_heads = nn.ModuleList([GatedAttentionMIL(D, attn_dim)
                                        for _ in range(n_findings)])
        self.heads = nn.ModuleList([nn.Linear(D, 1) for _ in range(n_findings)])
        self.coral_heads = nn.ModuleList([CoralHead(D) for _ in self.SEV_IDX])

    def forward_plane(self, x):  # x: (B, S, 3, H, W)
        B, S = x.shape[:2]
        # reshape (not view): non-contiguous tensors crash view() on CUDA
        h = self.backbone(x.reshape(B * S, *x.shape[2:])).reshape(B, S, -1)
        return torch.stack([mil(h)[0] for mil in self.mil_heads], dim=1)  # (B,12,D)

    def forward(self, planes, plane_mask):
        feats = torch.stack([self.forward_plane(planes[p]) for p in PLANES], dim=1)
        m = plane_mask.reshape(-1, 3, 1, 1)
        fused = (feats * m).sum(1) / m.sum(1).clamp(min=1e-6)  # (B,12,D)
        logits = torch.cat([self.heads[f](fused[:, f, :])
                            for f in range(N_FINDINGS)], dim=1)          # (B,12)
        coral = torch.stack([self.coral_heads[i](fused[:, self.SEV_IDX[i], :])
                             for i in range(len(self.SEV_IDX))], dim=1)  # (B,4,3)
        return logits, coral

    @property
    def enc(self):
        return self.backbone  # for frozen-BN helper


def freeze_bn_stats(model):
    # PLAN.md §6.2: encoder BN in eval mode (uses pretrained running stats).
    # Affine params still train. Call AFTER model.train() every epoch.
    n = 0
    for mod in model.enc.modules():
        if isinstance(mod, nn.modules.batchnorm._BatchNorm):
            mod.eval()
            n += 1
    return n


def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6


model = KneeEffNetCORAL()
print(f'params: {count_params(model):.1f}M')
# CPU shape probe before GPU
model.eval()
with torch.no_grad():
    _pb = {p: torch.randn(1, 4, 3, IMAGE_SIZE, IMAGE_SIZE) for p in PLANES}
    _pm = torch.ones(1, 3)
    _o, _c = model(_pb, _pm)
    print(f'cpu shape probe: in (1,4,3,{IMAGE_SIZE},{IMAGE_SIZE}) x3 -> '
          f'binary {tuple(_o.shape)}, coral {tuple(_c.shape)}')
    assert _o.shape == (1, N_FINDINGS) and _c.shape == (1, 4, N_THRESH)
    # rank consistency: P(>=mild) >= P(>=moderate) >= P(>=severe)
    _p = torch.sigmoid(_c)
    assert bool((_p[:, :, 0] >= _p[:, :, 1] - 1e-6).all()) and \
           bool((_p[:, :, 1] >= _p[:, :, 2] - 1e-6).all()), 'CORAL rank consistency broken'
print('model self-test: ALL GREEN (shapes + rank consistency)')
del model  # rebuild in training cell / probe


In [ ]:
# ================= TRAINING UTILS: AdamW groups, warmup+cosine, EMA, AMP, CORAL =================
def make_loader(uids, augment, batch_size=BATCH_SIZE, shuffle=True, num_workers=4):
    ds = KneeDataset(uids, store, label_pack, augment=augment)

    def collate(batch):
        out = {'uid': [b['uid'] for b in batch]}
        out['planes'] = {p: torch.stack([b['planes'][p] for b in batch]) for p in PLANES}
        for k in ['plane_mask', 'targets', 'loss_mask', 'weight', 'is_gold',
                  'coral_targets', 'coral_mask']:
            out[k] = torch.stack([b[k] for b in batch])
        return out

    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=num_workers, pin_memory=True, collate_fn=collate)


def masked_asl(logits, targets, loss_mask, sample_weight):
    el = asl_loss(logits, targets)
    per_sample = (el * loss_mask).sum(1) / loss_mask.sum(1).clamp(min=1e-6)
    return (per_sample * sample_weight).mean()


_CORAL_W = torch.tensor(CORAL_THRESH_W, dtype=torch.float32)  # [1.0, 0.5, 0.5]

def coral_loss(coral_logits, coral_targets, coral_mask):
    """Weighted-threshold CORAL BCE. All inputs MUST be fp32 (caller guarantees).
    coral_logits/targets: (B,4,3); coral_mask: (B,4).
    Threshold weights [1.0, 0.5, 0.5]: full weight on t0 (presence, 100%-validated),
    reduced on t1/t2 (grade granularity never expert-validated)."""
    bce = F.binary_cross_entropy_with_logits(coral_logits, coral_targets, reduction='none')
    w = _CORAL_W.to(coral_logits.device).view(1, 1, N_THRESH)
    per = (bce * w).sum(-1) / w.sum()            # (B,4)
    return (per * coral_mask).sum() / coral_mask.sum().clamp(min=1e-6)


def forward_loss(model, planes, plane_mask, targets, loss_mask, sample_weight,
                 coral_targets, coral_mask):
    """One canonical forward+loss path: backbone forward in AMP for speed,
    ALL losses in fp32 (v2.2 NaN lesson: fp16 sigmoid saturates -> inf -> NaN)."""
    with torch.cuda.amp.autocast(enabled=USE_AMP):
        logits, coral = model(planes, plane_mask)
    with torch.cuda.amp.autocast(enabled=False):
        main = masked_asl(logits.float(), targets, loss_mask, sample_weight)
        cor = coral_loss(coral.float(), coral_targets, coral_mask)
        loss = main + CORAL_LAMBDA * cor
    return loss, logits, coral, main, cor


@torch.no_grad()
def evaluate(model, loader):
    # Scored metric is ALWAYS the 12 binary heads (gold-58 labels are binary).
    # CORAL heads are auxiliary and never scored.
    model.eval()
    ys, ps, gm = [], [], []
    for b in loader:
        planes = {p: v.to(DEVICE, non_blocking=True) for p, v in b['planes'].items()}
        logits, _ = model(planes, b['plane_mask'].to(DEVICE))
        ys.append(b['targets'].numpy())
        ps.append(torch.sigmoid(logits).float().cpu().numpy())
        gm.append(b['is_gold'].numpy())
    ys = np.concatenate(ys); ps = np.concatenate(ps)
    gm = np.concatenate(gm).astype(bool)
    yb = (ys >= 0.5).astype(int)
    macro, per = macro_auc(yb, ps)
    macro_g, per_g = (None, {})
    if gm.sum() >= 4:
        macro_g, per_g = macro_auc(yb[gm], ps[gm])
    return macro, per, macro_g


def get_param_groups(model, lr_backbone=LR_BACKBONE, lr_head=LR_HEAD,
                     lr_coral=LR_CORAL, weight_decay=WEIGHT_DECAY):
    # 3 tiers x decay/no-decay. No decay on biases, norms, 1-D.
    # Tiers: backbone (converged, gentle) | binary heads (trained, fine-tune) |
    #        coral heads (fresh params, full LR).
    bb, hd, co = [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith('backbone.'):
            bb.append((name, p))
        elif 'coral' in name:
            co.append((name, p))
        else:
            hd.append((name, p))
    assert co, 'FATAL: no coral params found — model class mismatch?'
    assert bb and hd, 'FATAL: backbone/head param routing broken'

    def split(params):
        decay = [p for n, p in params if p.dim() > 1]
        no_decay = [p for n, p in params if p.dim() <= 1]
        return decay, no_decay

    groups = []
    for tier, lr in [(bb, lr_backbone), (hd, lr_head), (co, lr_coral)]:
        d, nd = split(tier)
        groups.append({'params': d, 'lr': lr, 'weight_decay': weight_decay})
        groups.append({'params': nd, 'lr': lr, 'weight_decay': 0.0})
    print(f'  param groups: bb={len(bb)} head={len(hd)} coral={len(co)} params | '
          f'lr {lr_backbone:.0e}/{lr_head:.0e}/{lr_coral:.0e}')
    return groups


def make_scheduler(opt, epochs, steps_per_epoch, warmup_frac=WARMUP_FRAC):
    total_steps = epochs * steps_per_epoch
    warmup_steps = max(1, int(warmup_frac * total_steps))

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(warmup_steps)
        prog = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))

    return torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda), total_steps, warmup_steps


class EMA:
    """Exponential moving average of model weights (decay 0.998). CPU shadow."""
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach().cpu(),
                                                     alpha=1.0 - self.decay)

    def state_dict(self):
        return {k: v.clone() for k, v in self.shadow.items()}


def average_state_dicts(sds):
    # PLAN.md §6.4: manual SWA over EMA snapshots
    out = {}
    for k, v in sds[-1].items():
        if v.dtype.is_floating_point:
            out[k] = torch.stack([sd[k].float() for sd in sds]).mean(0).to(v.dtype)
        else:
            out[k] = v.clone()  # e.g. BN num_batches_tracked
    return out


def save_ckpt(path, model, opt, epoch, best, ema_sd=None, swa_sd=None):
    # float(best): torch>=2.6 weights_only loader rejects np.float64
    payload = {'epoch': epoch, 'best': float(best),
               'model': model.state_dict(),
               'opt': opt.state_dict() if opt is not None else None}
    if ema_sd is not None:
        payload['ema'] = ema_sd
    if swa_sd is not None:
        payload['swa'] = swa_sd
    torch.save(payload, path)
    print(f'  saved {path}')


def load_base_ckpt(path, model):
    """Load the 0.89 baseline ckpt into backbone+binary heads (strict=False:
    coral heads are new). FAILS LOUD unless the key sets match exactly."""
    assert os.path.exists(path), f'FATAL: base ckpt not found: {path}'
    ck = torch.load(path, map_location='cpu')
    sd = ck['model'] if isinstance(ck, dict) and 'model' in ck else ck
    missing, unexpected = model.load_state_dict(sd, strict=False)
    exp_missing = sorted(k for k in model.state_dict() if k.startswith('coral_heads.'))
    bad_missing = sorted(set(missing) - set(exp_missing))
    assert not bad_missing, f'FATAL: ckpt missing non-coral keys: {bad_missing}'
    assert not unexpected, f'FATAL: ckpt has unexpected keys: {unexpected}'
    print(f"  base ckpt loaded: {path} (epoch {ck.get('epoch')}, best {ck.get('best')})")
    print(f'  restored {len(sd)} tensors (backbone + binary heads); '
          f'{len(exp_missing)} coral params freshly initialized')


In [ ]:
# ================= PROBE MODE: full pipeline on tiny subset =================
# Gate: this cell must print ALL GREEN before any full run.
if PROBE:
    print('=== PROBE: coral forward + loss + AMP + EMA + base-ckpt load + rank consistency ===')
    pm = KneeEffNetCORAL().to(DEVICE)
    load_base_ckpt(BASE_CKPT, pm)   # fail-fast: exact key-set match required
    popt = torch.optim.AdamW(get_param_groups(pm), lr=LR_BACKBONE)
    pscaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    pema = EMA(pm)
    ptr = make_loader(train_uids, augment=True, batch_size=2, shuffle=True, num_workers=0)
    pva = make_loader(val_uids, augment=False, batch_size=2, shuffle=False, num_workers=0)
    b = next(iter(ptr))
    print(f"  batch planes: {tuple(b['planes']['axial'].shape)} "
          f"(expect (2,{N_SLICES},3,{IMAGE_SIZE},{IMAGE_SIZE}))")
    assert b['planes']['axial'].dtype == torch.float32, 'FATAL: float64 leak in dataset'
    assert tuple(b['coral_targets'].shape) == (2, 4, N_THRESH), \
        f'FATAL: bad coral target shape {tuple(b["coral_targets"].shape)}'
    print(f"  coral targets: {tuple(b['coral_targets'].shape)} "
          f"| mask sum {b['coral_mask'].sum().item():.0f}/8 (expect 8 if all graded)")

    # --- DataParallel: SKIPPED (v1.1) ---
    # DP corrupts the CUDA context on T4s even when the exception is caught.
    # CoAtNet proved this twice; EffNet v1 probe confirmed it a third time.
    # Single-GPU only. No test needed.
    print('  DataParallel: SKIPPED (known unsafe on T4, single-GPU only)')

    # --- train step with AMP + grad accum + EMA ---
    planes = {p: v.to(DEVICE) for p, v in b['planes'].items()}
    pm.train()
    _nbn = freeze_bn_stats(pm)
    print(f'  frozen BN modules: {_nbn}')
    t0 = time.time()
    popt.zero_grad()
    loss, logits, coral, main, cor = forward_loss(
        pm, planes, b['plane_mask'].to(DEVICE),
        b['targets'].to(DEVICE), b['loss_mask'].to(DEVICE),
        b['weight'].to(DEVICE), b['coral_targets'].to(DEVICE),
        b['coral_mask'].to(DEVICE))
    assert logits.shape == (2, N_FINDINGS), f'bad logits shape {logits.shape}'
    assert coral.shape == (2, 4, N_THRESH), f'bad coral shape {coral.shape}'
    assert torch.isfinite(loss) and torch.isfinite(cor), 'FATAL: NaN loss on probe'
    assert cor.item() >= 0, 'FATAL: negative coral loss'
    # rank consistency on REAL forward output
    _p = torch.sigmoid(coral)
    assert bool((_p[:, :, 0] >= _p[:, :, 1] - 1e-5).all()) and \
           bool((_p[:, :, 1] >= _p[:, :, 2] - 1e-5).all()), 'FATAL: CORAL rank broken'
    print(f'  forward+AMP: binary {tuple(logits.shape)}, coral {tuple(coral.shape)} '
          f'in {time.time()-t0:.1f}s | loss {loss.item():.4f} '
          f'(main {main.item():.4f} + λ·coral {cor.item():.4f}) | rank-consistent OK')
    pscaler.scale(loss).backward()
    pscaler.unscale_(popt)
    gn = torch.nn.utils.clip_grad_norm_(pm.parameters(), GRAD_CLIP)
    # coral heads must have received gradients (fresh params, nothing frozen)
    _cgn = [p.grad.abs().sum().item() for n, p in pm.named_parameters()
            if 'coral' in n and p.grad is not None]
    assert _cgn and all(g > 0 for g in _cgn), 'FATAL: coral heads got no gradient'
    pscaler.step(popt); pscaler.update()
    pema.update(pm)
    print(f'  backward+step+EMA OK (grad norm {gn:.2f}; coral grads flowing)')
    if torch.cuda.is_available():
        print(f'  gpu mem: {torch.cuda.max_memory_allocated(0)/1e9:.2f} GB peak (T4 = 14.56)')

    # --- baseline sanity: loaded 0.89 ckpt on the val fold ---
    macro, per, macro_g = evaluate(pm, pva)
    print(f'  BASELINE (loaded 0.89 ckpt) val macro: {macro:.4f}' if macro else '  val macro: n/a',
          f'| gold-macro {macro_g:.4f} (target to beat: {BASELINE_GOLD_MACRO})' if macro_g else '')
    save_ckpt(f'{CKPT_DIR}/probe.pt', pm, popt, 0, macro or 0, ema_sd=pema.state_dict())
    print('PROBE: ALL GREEN — full run cleared')
else:
    print('PROBE=False — skipping probe (only do this after a green probe)')


In [ ]:
# ================= TRAINING: 5-epoch CORAL fine-tune from the 0.89 ckpt =================
def train_coral(uids_tr, uids_va, epochs=EPOCHS):
    model = KneeEffNetCORAL().to(DEVICE)
    # Single-GPU (CoAtNet proved DP-unsafe; EffNet DP test is probe-only).
    load_base_ckpt(BASE_CKPT, model)   # fail-fast: backbone+binary heads must match exactly
    opt = torch.optim.AdamW(get_param_groups(model), lr=LR_BACKBONE)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    ema = EMA(model, decay=EMA_DECAY)
    ema_snaps = []  # ring buffer of last SWA_LAST EMA snapshots (CPU)

    # Baseline sanity BEFORE training: the loaded ckpt is the 0.89 run.
    va_loader = make_loader(uids_va, augment=False, shuffle=False)
    _m0, _, _g0 = evaluate(model, va_loader)
    print(f'  BASELINE ckpt: val macro {_m0:.4f}' if _m0 else '  BASELINE: val n/a',
          f'| gold-macro {_g0:.4f} (target: {BASELINE_GOLD_MACRO})' if _g0 else '')

    tr_loader = make_loader(uids_tr, augment=True)
    steps_per_epoch = math.ceil(len(uids_tr) / BATCH_SIZE)
    opt_steps_per_epoch = math.ceil(steps_per_epoch / ACCUM_STEPS)
    sched, total_steps, warmup_steps = make_scheduler(opt, epochs, opt_steps_per_epoch)
    print(f'train: {len(uids_tr)} studies / val: {len(uids_va)} | '
          f'{steps_per_epoch} batches/ep x {epochs} eps | '
          f'{opt_steps_per_epoch} opt-steps/ep | warmup {warmup_steps} steps | '
          f'total {total_steps} opt-steps')
    print(f'AMP: {USE_AMP} | EMA decay: {EMA_DECAY} | freeze BN: {FREEZE_BN} | '
          f'coral λ: {CORAL_LAMBDA}')

    best, best_gold = -1, -1
    for ep in range(epochs):
        model.train()
        if FREEZE_BN:
            freeze_bn_stats(model)  # §6.2: BN stays in eval (pretrained stats)
        t_ep, tot, tot_main, tot_cor, n = time.time(), 0.0, 0.0, 0.0, 0
        opt.zero_grad()
        for si, b in enumerate(tr_loader):
            planes = {p: v.to(DEVICE, non_blocking=True) for p, v in b['planes'].items()}
            loss, _, _, main, cor = forward_loss(
                model, planes, b['plane_mask'].to(DEVICE),
                b['targets'].to(DEVICE), b['loss_mask'].to(DEVICE),
                b['weight'].to(DEVICE), b['coral_targets'].to(DEVICE),
                b['coral_mask'].to(DEVICE))
            if not torch.isfinite(loss):
                raise RuntimeError(f'KILL GATE: NaN/Inf loss at ep{ep} step{si}')
            scaler.scale(loss / ACCUM_STEPS).backward()
            if (si + 1) % ACCUM_STEPS == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad()
                ema.update(model)
                sched.step()
            tot += loss.item(); tot_main += main.item(); tot_cor += cor.item(); n += 1
        # leftover accum
        if n % ACCUM_STEPS:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad()
            ema.update(model)
            sched.step()

        # EMA snapshot at epoch end (CPU ring buffer)
        ema_snaps.append(ema.state_dict())
        ema_snaps = ema_snaps[-SWA_LAST:]

        macro, per, macro_g = evaluate(model, va_loader)
        dt = time.time() - t_ep
        lr_now = opt.param_groups[0]['lr']
        print(f'  ep{ep+1}/{epochs} | loss {tot/n:.4f} (main {tot_main/n:.4f} + '
              f'λ·coral {tot_cor/n:.4f}) | lr {lr_now:.2e} | '
              f'val macro {macro:.4f}' if macro else f'  ep{ep+1} val n/a',
              f'| gold-macro {macro_g:.4f}' if macro_g else '', f'| {dt:.0f}s',
              flush=True)
        improved = macro is not None and macro > best
        if improved:
            best = macro
            best_gold = macro_g if macro_g else -1
            save_ckpt(f'{CKPT_DIR}/coral_best.pt', model, opt, ep, best,
                      ema_sd=ema.state_dict())
        save_ckpt(f'{CKPT_DIR}/coral_last.pt', model, opt, ep, best,
                  ema_sd=ema.state_dict())
    return f'{CKPT_DIR}/coral_best.pt', best, best_gold, ema_snaps


if not PROBE:
    best_ckpt, best_val, best_gold, ema_snaps = train_coral(train_uids, val_uids)
    print(f'TRAINING done: best val macro AUC {best_val:.4f} -> {best_ckpt}')
    print(f'CORAL probe verdict: gold-macro {best_gold:.4f} vs baseline {BASELINE_GOLD_MACRO}')
    if best_gold > BASELINE_GOLD_MACRO:
        print('>>> GO: CORAL improved gold macro — schedule the full CORAL run / keep the variant.')
    else:
        print('>>> NO-GO: CORAL did not beat the baseline — drop it, no further spend.')
else:
    print('(probe mode: training skipped)')


In [ ]:
# ================= SWA: average last 3 EMA snapshots (PLAN.md §6.4) =================
if not PROBE:
    assert 'ema_snaps' in globals() and len(ema_snaps) == SWA_LAST, \
        f'FATAL: need {SWA_LAST} EMA snapshots, have {len(ema_snaps) if "ema_snaps" in globals() else 0}'
    print(f'averaging {len(ema_snaps)} EMA snapshots...')
    swa_sd = average_state_dicts(ema_snaps)
    # sanity: SWA weights differ from last snapshot but are close
    _k = next(k for k, v in swa_sd.items() if v.dtype.is_floating_point)
    _diff = (swa_sd[_k] - ema_snaps[-1][_k]).abs().mean().item()
    print(f'  mean abs diff vs last EMA snapshot ({_k}): {_diff:.2e} (expect small, nonzero)')

    swa_model = KneeEffNetCORAL().to(DEVICE)
    swa_model.load_state_dict(swa_sd)
    va_loader = make_loader(val_uids, augment=False, shuffle=False)
    macro_swa, per_swa, macro_g_swa = evaluate(swa_model, va_loader)
    print(f'  SWA val macro: {macro_swa:.4f}' if macro_swa else '  SWA val macro: n/a',
          f'| gold-macro {macro_g_swa:.4f}' if macro_g_swa else '')

    torch.save({'swa': swa_sd, 'val_macro': float(macro_swa or -1)},
               f'{CKPT_DIR}/coral_swa.pt')
    print(f'\nFINAL ARTIFACTS in {CKPT_DIR}/: coral_best.pt, coral_last.pt, coral_swa.pt')
    print('CORAL probe verdict above compares coral_best.pt gold-macro vs the 0.8628 baseline.')
else:
    print('(probe mode: SWA skipped)')
